In [27]:
from pathlib import Path
import pandas as pd

# ---------------------------
# User-configurable paths
# ---------------------------
manifest_csv = Path("table/runs_manifest_20260409_042243.csv")
evals_root = Path("evals")

# ---------------------------
# Table config
# ---------------------------
CONFIG_ORDER = ["CRW", "EE", "POI", "LPOI"]

MODEL_LABELS = {
    "rule-based": "Rule-based",
    "ppo": "PPO",
    "sac": "SAC",
    "dqn": "DQN",
}

MODEL_ORDER = ["rule-based", "dqn", "ppo", "sac"]

# (output_key, latex_label, source_column, transform_fn)
METRICS = [
    ("reward", "Total reward", "reward", lambda s: s),
    ("monitoring_reward", "Monitoring reward", "r_dist", lambda s: s + 1),
    ("disturbance_penalty", "Disturbance penalty", "p_disturbance", lambda s: s),
]

AGENT_CSVS = {
    "ppo": "ppo_ep.csv",
    "sac": "sac_ep.csv",
    "dqn": "dqn_ep.csv",
}

RULE_BASED_CSV = "CentroidStandoff_ep.csv"

# ---------------------------
# Helpers
# ---------------------------
def fmt_mean_std(series: pd.Series) -> str:
    mean = series.mean()
    std = series.std(ddof=1)
    return f"${mean:.3f} \\pm {std:.3f}$"

def read_eval_csv(csv_path: Path) -> pd.DataFrame:
    df = pd.read_csv(csv_path, comment="#")
    required_cols = [source_col for _, _, source_col, _ in METRICS]
    missing = [col for col in required_cols if col not in df.columns]
    if missing:
        raise ValueError(f"{csv_path} is missing required columns: {missing}")
    return df

def summarize_df(df: pd.DataFrame) -> dict:
    return {
        output_key: fmt_mean_std(transform_fn(df[source_col]))
        for output_key, _, source_col, transform_fn in METRICS
    }

# ---------------------------
# Load manifest
# ---------------------------
manifest = pd.read_csv(manifest_csv)

required_manifest_cols = {"config", "agent", "eval_name"}
missing_manifest_cols = required_manifest_cols - set(manifest.columns)
if missing_manifest_cols:
    raise ValueError(f"Manifest is missing required columns: {sorted(missing_manifest_cols)}")

# ---------------------------
# Aggregate results
# ---------------------------
results = {}

# PPO / SAC from agent-specific files
for _, row in manifest.iterrows():
    config = str(row["config"]).strip()
    agent = str(row["agent"]).strip().lower()
    eval_name = str(row["eval_name"]).strip()

    if agent not in AGENT_CSVS:
        raise ValueError(f"Unknown agent type: {agent}")

    csv_path = evals_root / eval_name / AGENT_CSVS[agent]
    if not csv_path.exists():
        raise FileNotFoundError(f"Missing eval CSV for {agent}: {csv_path}")

    df = read_eval_csv(csv_path)

    results.setdefault(agent, {})
    results[agent][config] = summarize_df(df)

# Rule-based from CentroidStandoff_ep.csv
# Since every eval folder for a given config contains the same rule-based file,
# use the first manifest row for each config.
results["rule-based"] = {}

for config in CONFIG_ORDER:
    matching = manifest[manifest["config"].astype(str).str.strip() == config]
    if matching.empty:
        continue

    eval_name = str(matching.iloc[0]["eval_name"]).strip()
    csv_path = evals_root / eval_name / RULE_BASED_CSV

    if not csv_path.exists():
        raise FileNotFoundError(f"Missing rule-based CSV for {config}: {csv_path}")

    df = read_eval_csv(csv_path)
    results["rule-based"][config] = summarize_df(df)

# ---------------------------
# Build LaTeX table
# ---------------------------
lines = []
lines.append(r"\begin{table}[!ht]")
lines.append(r"\centering")
lines.append(r"\setlength{\tabcolsep}{4pt}")
lines.append(
    r"\caption{Performance comparison across behaviors. Results are reported as mean $\pm$ standard deviation across evaluation runs of \(N=100\) repetitions.}"
)
lines.append(r"\label{tab:behavior_results_full}")
lines.append(r"\begin{tabular}{llcccc}")
lines.append(r"\hline \hline")
lines.append(r"\textbf{Model} & \textbf{Metric} & \textbf{CRW} & \textbf{EE} & \textbf{POI} & \textbf{LPOI} \\")
lines.append(r"\hline \hline")

models_present = [m for m in MODEL_ORDER if m in results and results[m]]

for mi, model_key in enumerate(models_present):
    model_block = results[model_key]
    model_label = MODEL_LABELS[model_key]

    for metric_idx, (output_key, metric_label, _, _) in enumerate(METRICS):
        row_cells = []
        row_cells.append(model_label if metric_idx == 0 else "")
        row_cells.append(metric_label)

        for config in CONFIG_ORDER:
            value = model_block.get(config, {}).get(output_key, "--")
            row_cells.append(value)

        lines.append(" & ".join(row_cells) + r" \\")
        lines.append("")

    if mi < len(models_present) - 1:
        lines.append(r"\midrule")

lines.append(r"\hline \hline")
lines.append(r"\end{tabular}")
lines.append(r"\end{table}")

latex_table = "\n".join(lines)
print(latex_table)

\begin{table}[!ht]
\centering
\setlength{\tabcolsep}{4pt}
\caption{Performance comparison across behaviors. Results are reported as mean $\pm$ standard deviation across evaluation runs of \(N=100\) repetitions.}
\label{tab:behavior_results_full}
\begin{tabular}{llcccc}
\hline \hline
\textbf{Model} & \textbf{Metric} & \textbf{CRW} & \textbf{EE} & \textbf{POI} & \textbf{LPOI} \\
\hline \hline
Rule-based & Total reward & $0.770 \pm 0.025$ & $0.827 \pm 0.024$ & $0.768 \pm 0.023$ & $0.761 \pm 0.018$ \\

 & Monitoring reward & $0.473 \pm 0.010$ & $0.495 \pm 0.008$ & $0.469 \pm 0.012$ & $0.453 \pm 0.012$ \\

 & Disturbance penalty & $0.207 \pm 0.005$ & $0.196 \pm 0.007$ & $0.205 \pm 0.003$ & $0.195 \pm 0.003$ \\

\midrule
DQN & Total reward & $0.776 \pm 0.020$ & $0.864 \pm 0.012$ & $0.868 \pm 0.015$ & $0.823 \pm 0.013$ \\

 & Monitoring reward & $0.475 \pm 0.039$ & $0.492 \pm 0.027$ & $0.481 \pm 0.017$ & $0.504 \pm 0.011$ \\

 & Disturbance penalty & $0.177 \pm 0.027$ & $0.153 \pm 0.015$ & $0

In [24]:
from pathlib import Path
import pandas as pd

manifest_csv = Path("table/replay_eval_manifest_20260409_183629.csv")
evals_root = Path("evals/replay")

SPECIES_ORDER = ["JACKALS", "PIGEONS", "SPUR"]
TRAINING_ORDER = ["BEST", "GPS"]
MODEL_ORDER = ["ppo", "sac"]

SPECIES_LABELS = {
    "JACKALS": "Jackal",
    "PIGEONS": "Pigeon",
    "SPUR": "Spur-winged lapwing",
}

MODEL_LABELS = {
    "ppo": "PPO",
    "sac": "SAC",
}

BEST_TRAINING_LABELS = {
    "JACKALS": "LPOI",
    "PIGEONS": "LPOI",
    "SPUR": "EE",
}

# (output_key, row_label, source_column, transform)
METRICS = [
    ("reward", "Total reward", "reward", lambda s: s),
    ("monitoring_reward", "Monitoring reward", "r_dist", lambda s: s + 1),
    ("disturbance_penalty", "Disturbance penalty", "p_disturbance", lambda s: s),
]

def fmt_mean_std(series: pd.Series) -> str:
    return f"${series.mean():.3f} \\pm {series.std(ddof=1):.3f}$"

manifest = pd.read_csv(manifest_csv)

results = {}

for _, row in manifest.iterrows():
    cfg = str(row["config"]).strip()
    agent = str(row["agent"]).strip().lower()
    eval_name = str(row["eval_name"]).strip()

    if cfg.startswith("JACKALS"):
        species = "JACKALS"
    elif cfg.startswith("PIGEONS"):
        species = "PIGEONS"
    elif cfg.startswith("SPUR"):
        species = "SPUR"
    else:
        continue

    training_data = "GPS" if "GPS" in cfg else "BEST"

    ep_csv = evals_root / eval_name / f"{agent}_ep.csv"
    if not ep_csv.exists():
        raise FileNotFoundError(ep_csv)

    df = pd.read_csv(ep_csv, comment="#")

    results.setdefault(training_data, {})
    results[training_data].setdefault(agent, {})
    results[training_data][agent][species] = {
        out_key: fmt_mean_std(transform(df[source_col]))
        for out_key, _, source_col, transform in METRICS
    }

lines = []
lines.append(r"\begin{table}[!ht]")
lines.append(r"\centering")
lines.append(r"\setlength{\tabcolsep}{4pt}")
lines.append(r"\caption{Replay evaluation performance across species, training data, and models. Results are reported as mean $\pm$ standard deviation across 100 evaluation episodes.}")
lines.append(r"\label{tab:replay_results_all_species}")
lines.append(r"\begin{tabular}{llcccc}")
lines.append(r"\hline \hline")
lines.append(r"\textbf{Training} & \textbf{Model} & \textbf{Metric} & \textbf{Jackal} & \textbf{Pigeon} & \textbf{Spur-winged lapwing} \\")
lines.append(r"\hline \hline")

row_blocks = []
for training_data in TRAINING_ORDER:
    training_label = {
        "JACKALS": "GPS" if training_data == "GPS" else BEST_TRAINING_LABELS["JACKALS"],
        "PIGEONS": "GPS" if training_data == "GPS" else BEST_TRAINING_LABELS["PIGEONS"],
        "SPUR": "GPS" if training_data == "GPS" else BEST_TRAINING_LABELS["SPUR"],
    }

    # single label for the row block
    block_label = "GPS" if training_data == "GPS" else "Best-fit"

    for agent in MODEL_ORDER:
        for metric_idx, (out_key, metric_label, _, _) in enumerate(METRICS):
            training_cell = rf"\multirow{{6}}{{*}}{{{block_label}}}" if (agent == MODEL_ORDER[0] and metric_idx == 0) else ""
            model_cell = rf"\multirow{{3}}{{*}}{{{MODEL_LABELS[agent]}}}" if metric_idx == 0 else ""

            row = [
                training_cell,
                model_cell,
                metric_label,
            ]

            for species in SPECIES_ORDER:
                value = results.get(training_data, {}).get(agent, {}).get(species, {}).get(out_key, "--")
                row.append(value)

            lines.append(" & ".join(row) + r" \\")
        lines.append("")

    if training_data != TRAINING_ORDER[-1]:
        lines.append(r"\midrule")

lines.append(r"\hline \hline")
lines.append(r"\end{tabular}")
lines.append(r"\end{table}")

latex_table = "\n".join(lines)
print(latex_table)

\begin{table}[!ht]
\centering
\setlength{\tabcolsep}{4pt}
\caption{Replay evaluation performance across species, training data, and models. Results are reported as mean $\pm$ standard deviation across 100 evaluation episodes.}
\label{tab:replay_results_all_species}
\begin{tabular}{llcccc}
\hline \hline
\textbf{Training} & \textbf{Model} & \textbf{Metric} & \textbf{Jackal} & \textbf{Pigeon} & \textbf{Spur-winged lapwing} \\
\hline \hline
\multirow{6}{*}{Best-fit} & \multirow{3}{*}{PPO} & Total reward & $0.878 \pm 0.087$ & $0.867 \pm 0.039$ & $0.841 \pm 0.167$ \\
 &  & Monitoring reward & $0.541 \pm 0.031$ & $0.548 \pm 0.078$ & $0.470 \pm 0.088$ \\
 &  & Disturbance penalty & $0.201 \pm 0.027$ & $0.207 \pm 0.077$ & $0.141 \pm 0.044$ \\

 & \multirow{3}{*}{SAC} & Total reward & $0.922 \pm 0.008$ & $0.919 \pm 0.009$ & $0.928 \pm 0.011$ \\
 &  & Monitoring reward & $0.566 \pm 0.021$ & $0.565 \pm 0.010$ & $0.517 \pm 0.025$ \\
 &  & Disturbance penalty & $0.192 \pm 0.017$ & $0.190 \pm 0.008$ 